# Week 07 Lab Work

**Dataset:** Adult / Census Income (UCI Machine Learning Repository)

1. Load the chosen and pre-processed dataset.
2. Split it into training (70%), testing (15%), and validation (15%).
3. Apply the three variants of Decision Tree and check the accuracy on the test set.

In [1]:
import os
import urllib.request

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

cols = ["age", "workclass", "fnlwgt", "education", "education-num",
        "marital-status", "occupation", "relationship", "race", "sex",
        "capital-gain", "capital-loss", "hours-per-week", "native-country", "income"]


def get_file(data_filename):
    for base in [".", ".."]:
        if not os.path.isdir(base):
            continue
        for root, dirs, files in os.walk(base):
            if data_filename in files:
                return os.path.join(root, data_filename)

    print(f"'{data_filename}' not found in the workspace. Downloading it dynamically...")
    url = f"https://archive.ics.uci.edu/ml/machine-learning-databases/adult/{data_filename}"
    urllib.request.urlretrieve(url, data_filename)
    return data_filename


# 1. Load and pre-process the data
path = get_file("adult.data")
print(f"Loading data from: {path}")
df = pd.read_csv(path, header=None, names=cols, skipinitialspace=True, na_values="?")
df = df.dropna().drop_duplicates().reset_index(drop=True)
df["income"] = (df["income"] == ">50K").astype(int)
df = pd.get_dummies(df, drop_first=True)

print("Shape:", df.shape)
print("Missing cells:", df.isnull().sum().sum())
df.head()

'adult.data' not found in the workspace. Downloading it dynamically...


Loading data from: adult.data
Shape: (30139, 97)
Missing cells: 0


,age,fnlwgt,education-num,capital-gain,capital-loss,hours-per-week,income,workclass_Local-gov,workclass_Private,workclass_Self-emp-inc,...,native-country_Portugal,native-country_Puerto-Rico,native-country_Scotland,native-country_South,native-country_Taiwan,native-country_Thailand,native-country_Trinadad&Tobago,native-country_United-States,native-country_Vietnam,native-country_Yugoslavia
0,39,77516,13,2174,0,40,0,False,False,False,...,False,False,False,False,False,False,False,True,False,False
1,50,83311,13,0,0,13,0,False,False,False,...,False,False,False,False,False,False,False,True,False,False
2,38,215646,9,0,0,40,0,False,True,False,...,False,False,False,False,False,False,False,True,False,False
3,53,234721,7,0,0,40,0,False,True,False,...,False,False,False,False,False,False,False,True,False,False
4,28,338409,13,0,0,40,0,False,True,False,...,False,False,False,False,False,False,False,False,False,False


## 2. Train, test and validation split

In [2]:
X = df.drop(columns=["income"])
y = df["income"]

# 70% train, 30% temp
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42
)

# split the remaining 30% into 15% test / 15% validation
X_test, X_val, y_test, y_val = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42
)

print(f"Train set: {len(X_train)} rows ({len(X_train)/len(df):.1%})")
print(f"Test set:  {len(X_test)} rows ({len(X_test)/len(df):.1%})")
print(f"Val set:   {len(X_val)} rows ({len(X_val)/len(df):.1%})")

Train set: 21097 rows (70.0%)
Test set:  4521 rows (15.0%)
Val set:   4521 rows (15.0%)


## 3. Three variants of Decision Tree

In [3]:
models = {
    "Decision Tree (Gini)": DecisionTreeClassifier(criterion="gini", random_state=42),
    "Decision Tree (Entropy)": DecisionTreeClassifier(criterion="entropy", random_state=42),
    "Decision Tree (Entropy, max_depth=8, pruned)": DecisionTreeClassifier(
        criterion="entropy", max_depth=8, random_state=42),
}

results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    results[name] = accuracy_score(y_test, model.predict(X_test))
    print(f"{name}: {results[name]:.4f} ({results[name]:.2%})")

results_df = pd.DataFrame({"Decision Tree Variant": list(results.keys()),
                           "Test Accuracy": list(results.values())})
results_df["Test Accuracy (%)"] = (results_df["Test Accuracy"] * 100).round(2)
results_df

Decision Tree (Gini): 0.8120 (81.20%)


Decision Tree (Entropy): 0.8142 (81.42%)
Decision Tree (Entropy, max_depth=8, pruned): 0.8456 (84.56%)


,Decision Tree Variant,Test Accuracy,Test Accuracy (%)
0,Decision Tree (Gini),0.811988,81.20
1,Decision Tree (Entropy),0.814200,81.42
2,"Decision Tree (Entropy, max_depth=8, pruned)",0.845609,84.56
